In [1]:
import sys
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# audited splits: train is the 257-row cleaned set
train_df = pd.read_excel("../data_splits/train.xlsx")
val_df   = pd.read_excel("../data_splits/val.xlsx")
test_df  = pd.read_excel("../data_splits/test.xlsx")

labels = sorted(train_df["Label"].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
for s in (train_df, val_df, test_df):
    s["label_id"] = s["Label"].map(label2id)

# RoBERTa tokenizes differently from DeBERTa — recompute truncation from raw Trace Content
def head_tail_truncate_ids(text, tokenizer, max_length=512):
    ids = tokenizer.encode(text, add_special_tokens=False)
    budget = max_length - 2
    if len(ids) <= budget:
        return text
    head = budget // 2
    tail = budget - head
    return tokenizer.decode(ids[:head] + ids[-tail:])

for split in (train_df, val_df, test_df):
    split["text_truncated"] = split["Trace Content"].apply(
        lambda t: head_tail_truncate_ids(t, tokenizer)
    )

print(f"\nTrain {len(train_df)} / Val {len(val_df)} / Test {len(test_df)}")
print(f"Label mapping: {label2id}")
print(f"Truncated in train: {(train_df['Trace Content'] != train_df['text_truncated']).sum()} / {len(train_df)}")

Python: 3.12.10
PyTorch: 2.6.0+cu124
CUDA available: True
Device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

e:\Projects\agent-failure-detection\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\beeya\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (579 > 512). Running this sequence through the model will result in indexing errors



Train 257 / Val 87 / Test 87
Label mapping: {'HALLUCINATION': 0, 'LOOP': 1, 'SUCCESS': 2, 'UNSAFE_EXECUTION': 3}
Truncated in train: 77 / 257


In [2]:
from datasets import Dataset

def to_hf(split_df):
    return Dataset.from_pandas(
        split_df[["text_truncated", "label_id"]].rename(
            columns={"text_truncated": "text", "label_id": "label"}
        )
    )

train_ds, val_ds, test_ds = to_hf(train_df), to_hf(val_df), to_hf(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding=False)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

print(train_ds)

Map:   0%|          | 0/257 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 257
})
